# Researching external stock sentiment with Adanos and vectorbt

This notebook shows how to use the Adanos Finance Sentiment API as an external confirmation layer in a vectorbt research workflow.

We will:

- fetch source-level sentiment for a basket of stocks
- build simple composites such as `average_buzz` and `bullish_avg`
- combine those values with a price trend filter from vectorbt
- rank candidates for deeper analysis

Replace `ADANOS_API_KEY` with your own key before running the notebook.


In [ ]:
import requests
import pandas as pd
import vectorbt as vbt

ADANOS_API_KEY = "YOUR_ADANOS_API_KEY"
TICKERS = ["TSLA", "NVDA", "AAPL", "MSFT"]
LOOKBACK_DAYS = 7

ADANOS_ENDPOINTS = {
    "reddit": "https://api.adanos.org/reddit/stocks/v1/stock/{ticker}",
    "x": "https://api.adanos.org/x/stocks/v1/stock/{ticker}",
    "news": "https://api.adanos.org/news/stocks/v1/stock/{ticker}",
    "polymarket": "https://api.adanos.org/polymarket/stocks/v1/stock/{ticker}",
}


In [ ]:
def classify_alignment(bullish_values):
    spread = max(bullish_values) - min(bullish_values)
    if spread <= 15:
        return "Aligned"
    if spread <= 30:
        return "Mixed"
    return "Wide divergence"


def fetch_source_signal(session, source, ticker):
    response = session.get(
        ADANOS_ENDPOINTS[source].format(ticker=ticker),
        params={"days": LOOKBACK_DAYS},
        timeout=15,
    )
    response.raise_for_status()
    payload = response.json()

    volume = payload.get("trade_count")
    if volume is None:
        volume = payload.get("mentions", 0)

    return {
        "source": source,
        "buzz": float(payload.get("buzz_score", 0.0)),
        "bullish_pct": float(payload.get("bullish_pct", 0.0)),
        "trend": str(payload.get("trend", "stable")).lower(),
        "volume": int(volume or 0),
    }


def build_sentiment_row(ticker):
    session = requests.Session()
    session.headers["X-API-Key"] = ADANOS_API_KEY

    signals = []
    for source in ADANOS_ENDPOINTS:
        try:
            signals.append(fetch_source_signal(session, source, ticker))
        except requests.RequestException as exc:
            print(f"Skipping {ticker} {source}: {exc}")

    if not signals:
        return None

    buzz_values = [signal["buzz"] for signal in signals]
    bullish_values = [signal["bullish_pct"] for signal in signals]
    trends = [signal["trend"] for signal in signals]

    return {
        "ticker": ticker,
        "coverage": len(signals),
        "average_buzz": round(sum(buzz_values) / len(buzz_values), 1),
        "bullish_avg": round(sum(bullish_values) / len(bullish_values), 1),
        "rising_sources": sum(1 for trend in trends if trend == "rising"),
        "falling_sources": sum(1 for trend in trends if trend == "falling"),
        "source_alignment": classify_alignment(bullish_values),
    }


In [ ]:
rows = [build_sentiment_row(ticker) for ticker in TICKERS]
sentiment_df = pd.DataFrame([row for row in rows if row is not None]).set_index("ticker")
sentiment_df.sort_values(["average_buzz", "bullish_avg"], ascending=False)


In [ ]:
close = vbt.YFData.download(sentiment_df.index.tolist(), start="2024-01-01").get("Close")
ma50 = vbt.MA.run(close, 50).ma

price_snapshot = pd.DataFrame(
    {
        "close": close.iloc[-1],
        "ma50": ma50.iloc[-1],
        "one_month_return_pct": ((close.iloc[-1] / close.iloc[-21]) - 1).round(4) * 100,
    }
)
price_snapshot["above_50dma"] = price_snapshot["close"] > price_snapshot["ma50"]
price_snapshot


In [ ]:
research_df = sentiment_df.join(price_snapshot[["above_50dma", "one_month_return_pct"]])
research_df = research_df.sort_values(
    ["above_50dma", "average_buzz", "bullish_avg"],
    ascending=[False, False, False],
)
research_df


## How to use this output

A simple workflow is:

- focus first on symbols with `above_50dma = True`
- prefer higher `average_buzz` and `bullish_avg`
- be careful with `Wide divergence` across sources
- treat this as a ranking layer before deeper backtests or live strategy changes

This keeps Adanos in the role it fits best inside vectorbt: an external research input that helps prioritize which setups deserve attention.
